# 配电网可规划域：逐线路选型模型审核

本轮默认运行 **case33 的四条候选线路、两个型号**，核对紧凑 MILP/MISOCP、含规划变量的联合割与原 16 方案枚举。模型及联合割推导见 [docs/compact_planning.md](docs/compact_planning.md)。

Network 唯一保存物理数据和线路选项；model.py 建模；vertify.py 核对割和计算汇总；本 Notebook 组织求解与实验。紧凑求解器不读取完整方案表；本轮枚举仅作为独立对照。

COMPACT_REVIEW=False 可运行下方保留的小规模区域/AC 基准流程。此次审核不扩展到全网升级，也不把若干射线点的凸包称为整个规划域。


In [1]:
from importlib import import_module  # 按算例名称读取唯一网架配置。
from pathlib import Path  # 管理实验结果和源码路径。
from time import perf_counter  # 记录包含建模的实际求解时间。
from hashlib import sha256  # 保存本次实验的源码指纹。
import json  # 将基础记录写入唯一结果文件。
import nbformat  # Notebook 指纹只包含源单元。
import numpy as np  # 负荷、证书和割的数值运算。
import pandas as pd  # 从基础记录现场生成审核表。
import gurobipy as gp  # 初始化公共许可证及 AC 全局验证环境。
from scipy.spatial import HalfspaceIntersection  # 从有效外割恢复三维外域顶点。
from threadpoolctl import threadpool_limits  # 比较方法时统一矩阵运算线程数。
from IPython.display import display, IFrame  # 展示核对表和已有交互绘图。
from model import PlanningEquations, PlanningModel, PlanningSP, ACPowerFlow  # 正式模型仅有这四个主要类。
from region import simplex, halfspaces, polytope_vertices, clip_polytope, add_certificate, ResidualSearch  # 同方案凸域与并集几何操作。
from vertify import BenchmarkResult, region_membership, validate_power_flow, METHODS, METHOD_NAMES  # 独立 AC、区域标签和结果存取。
from vertify import verify_joint_cuts, compact_review_summary  # 小规模模型核对及联合割验证。
from plot import save_method_comparison, save_replay  # 图形只消费已有结果。

In [2]:
COMPACT_REVIEW = True             # 本轮先核对四条候选线路；False 运行原小规模区域基准
CASE = "case33bw"                 # 或 "four_bus_five_corridor"
PLANNING = True                    # False：固定最低费用方案
BUDGETS = (0., 1., 2., np.inf)      # 四节点用 (20000., 40000., 60000., np.inf)
DIVISIONS = 128                    # 每轴网格数；只影响事后评价
RADIAL_TOLERANCE = .002            # SOCP 连续内外域的径向停止精度
RECOMPUTE = True                   # False：明确读取本实验的 result.npz
OUTPUT = Path("results")/CASE/("planning" if PLANNING else "fixed")  # 规划与固定网架实验分别写入自己的唯一结果目录。

**网架与对照数据。** 四条候选支路 A、B、C、D 的两个型号分别为保持原状与同走廊并联一回，费用及 R/X 与原 16 方案完全相同。紧凑模型有 8 个逐线路型号变量，每条线路满足 $\sum_k x_{e,k}=1$。

以下 16 方案表只用于核对结果。完整区域基准逐方案调用同一个连续 SP 后取并集；紧凑主问题始终使用逐线路型号变量。

In [3]:
network = import_module(f"Network.{CASE}").network  # 网架、背景负荷、可选建设项目只从选定算例读取。
designs = network.designs if PLANNING else network.designs[:1]  # 方案表已按费用排序；固定网架模式只取最低费用方案。
budgets = BUDGETS if PLANNING else (np.inf,)  # 固定方案无需预算筛选，规划模式逐档比较允许方案的并集。
print(f"{CASE}: {len(designs)} 个建设方案；独立负荷节点 {designs[0].load_nodes}")  # 展示本次实际选择的网架和独立负荷节点。
display(pd.DataFrame([dict(方案=i, 建设变量=tuple(d.x), 费用=d.cost)  # 方案表只用于小规模独立核对和结果显示。
                      for i,d in enumerate(designs)]))  # 以编号引用各建设向量及费用，后续记录不重复保存。

case33bw: 16 个建设方案；独立负荷节点 (18, 25, 33)


,方案,建设变量,费用
0,0,"(0, 0, 0, 0)",0.0
1,1,"(0, 0, 0, 1)",1.0
2,2,"(0, 0, 1, 0)",1.0
3,3,"(0, 1, 0, 0)",1.0
4,4,"(0, 0, 1, 1)",2.0
5,5,"(0, 1, 0, 1)",2.0
6,6,"(0, 1, 1, 0)",2.0
7,7,"(1, 0, 0, 0)",2.0
8,8,"(0, 1, 1, 1)",3.0
9,9,"(1, 0, 0, 1)",3.0


**紧凑模型与联合割审核。** 主问题直接选择线路型号；固定 $x,p$ 后解连续 phase I，割为
$$a+b^\top p+d^\top x\ge0.$$
这里的循环没有完整方案索引，也没有人为迭代次数作为收敛依据。算法只接受经过原约束核验的可行状态，或具有对偶锥与有限界补偿的有效分离割。SOCP 证书仍是松弛模型的证书，不等同于 AC 等式认证。

负荷最大化查询用 MP 上界与相差 0.001 kW 的认证内点停止，记录两者；固定负荷的最小投资查询不移动负荷坐标。

In [4]:
def joint_benders(equations, *, power=None, budget=np.inf, direction=None, cuts=(), radial_gap_kw=1e-3):  # 一次非枚举的最低投资或负荷边界查询。
    problem = PlanningModel(equations,power=power,budget=budget,direction=direction,cuts_only=True)  # 查询只新建 MP，固定方程由外层复用。
    oracle = PlanningSP(equations)  # MP 与连续 SP 使用同一个方程对象。
    generated = []  # 本次新割只记录一次，调用者可跨查询复用。
    with problem.model:  # 完成本次查询后释放整数主问题。
        for cut in cuts:  # 加载同一物理模型已经证明有效的联合割。
            problem.add_cut(cut)  # 每条割同时约束负荷 p 和逐线路选型 x。
        while True:  # 以可行证书和目标界停止，不使用人为迭代次数宣称收敛。
            candidate = problem.solve()  # 得到全局最优外松弛候选和目标界。
            if candidate is None:  # 外松弛不可行即可证明原查询不可行。
                return None,generated,oracle.calls  # 保留此前取得的有效割及实际 SP 次数。
            check_power = candidate['p']  # 固定负荷查询必须检查原始负荷坐标。
            if power is None and check_power.sum()>0.:  # 边界查询在同方向内缩不超过指定 kW 间隙。
                check_power = check_power*max(0.,1.-radial_gap_kw/check_power.sum())  # 零或极小边界保持非负，不越过参数原点。
            result = oracle.solve(candidate['x'],check_power)  # 固定本轮整数选型，检查连续运行可行性。
            if result['feasible']:  # SP 已对不含 eta 的原始约束完成核验。
                candidate['state'] = result['state']  # 保存可行运行证书。
                candidate['p'] = check_power  # 返回实际获得认证的点。
                if power is None:  # 负荷目标使用内点；bound 仍为 MP 的全局上界。
                    candidate['objective'] = check_power.sum()  # 上下界差给出本方向优化误差。
                return candidate,generated,oracle.calls  # 当前查询完成，外层决定下一负荷或方向。
            if result['cut'] is None:  # 既无原查询证书也无分离割时，不能将该查询判为可行。
                raise RuntimeError('Planning query has no feasible certificate or separating cut')  # 保留未确定状态供调试。
            generated.append(result['cut'])  # 只保存新得到的严格分离割。
            problem.add_cut(result['cut'])  # 收紧 MP 后重新选择线路型号和负荷。

In [5]:
def review_compact(network, budgets, radial_gap_kw=1e-3):  # 比较四线路紧凑模型、联合割流程与独立枚举参考。
    """只核对当前四条候选线路；枚举参考和紧凑求解分开计时、使用不同潮流表示。"""
    from tests.reference import dispatch_support, dispatch_feasible  # 仅审核加载独立消元参考。
    startup = gp.Model()  # 公共许可证启动在两种方法的计时之外。
    startup.dispose()  # 公共许可证启动在两种方法的计时之外。
    rng = np.random.default_rng(20260920)  # 固定随机种子，使清理前后的查询集合可以复现。
    directions = np.vstack([np.eye(3),[1.,1.,1.],[1.,6.,1.],[2.,3.,1.],  # 选取轴向、等比例及有代表性的偏斜负荷方向。
                            rng.dirichlet([1.,3.,1.],size=6)])  # 补充六条可复现的非负随机方向。
    directions /= directions.sum(axis=1,keepdims=True)  # 射线系数和为 1，半径就是三个独立负荷之和，kW。
    plans = network.designs  # 枚举只出现在审核流程；PlanningModel/PlanningSP 不调用此属性。
    costs = np.array([d.cost for d in plans])  # 从唯一方案表读取预算筛选所需费用。
    record = dict(budgets=[None if np.isinf(b) else b for b in budgets],directions=directions,  # 记录审核使用的预算和方向。
                  design_choices=[d.x for d in plans],design_costs=costs,radial_gap_kw=radial_gap_kw,models={})  # 方案选型、投资及查询间隙各保存一份。
    for method in ('linear','socp'):  # 分别为线性和 SOCP 生成独立参考边界。
        start = perf_counter()  # 计时包括参考模型构建和求解。
        radii = np.array([[dispatch_support(d,method,np.ones(3),direction=a)['value']  # 对每个完整方案、每条方向独立求支持边界。
                          for a in directions] for d in plans])  # 16 个独立消元模型逐方案求边界。
        record['models'][method] = dict(enumerated_radii=radii,  # 保存原始逐方案半径，预算并集边界由其推导。
                                        seconds=dict(enumeration_boundary_seconds=perf_counter()-start))  # 记录参考边界查询的实测时间。
    bounds = np.max(record['models']['linear']['enumerated_radii'][:,:3],axis=0)  # 三条 LP 轴向参考边界决定随机负荷取值范围。
    points = [rng.random((32,3))*bounds]  # 均匀箱内样本，含域内及域外点。
    for data in record['models'].values():  # 同时使用两种物理模型的边界生成审核查询。
        for budget in budgets:  # 覆盖所有预算下的边界附近点。
            boundary = np.max(data['enumerated_radii'][costs<=budget],axis=0)  # 同方向的规划边界取预算内方案的最大半径。
            for scale in (.999,1.001):  # 显式检查 LP/SOCP 各预算边界的两侧，不只选容易的内部点。
                points.append(scale*boundary[:6,None]*directions[:6])  # 在边界两侧构造点，检验投资切换及区域归属。
    points = np.unique(np.round(np.vstack(points),8),axis=0)  # 仅去除重复查询点，所有方法共享同一集合。
    record['points'] = points  # 点坐标只保存一次，查询记录采用索引引用。
    for method,data in record['models'].items():  # 每种物理模型独立计时和核对。
        equations = PlanningEquations(network,method)  # 同一网架和物理模型的所有规划查询共用一次方程构造。
        start = perf_counter()  # 开始计量参考模型的最小投资计算。
        minimum = np.full(len(points),np.inf)  # 尚未发现可行方案的点记为无限投资。
        for design in plans:  # 按费用顺序检查全部参考方案。
            feasible = dispatch_feasible(design,method,points)  # 参考 LP 直接检查余量，SOCP 使用独立消元模型。
            minimum[feasible] = np.minimum(minimum[feasible],design.cost)  # 全部方案中取最小可行投资。
        data['enumerated_cost'] = minimum  # 保存参考最小投资，供四档预算共同使用。
        data['seconds']['enumeration_point_seconds'] = perf_counter()-start  # 记录参考投资查询时间，区分边界查询。
        start = perf_counter()  # 开始计量紧凑直接模型的相同投资查询。
        compact_cost, choices, radii = [], [], []  # 分别收集投资、实际选型及后续边界半径。
        for power in points:  # 逐点询问该负荷所需的最低建设费用。
            problem = PlanningModel(equations,power=power)  # 使用同一套规划方程，固定当前查询负荷。
            with problem.model:  # 查询后释放 Gurobi 模型，不保留重复运行状态。
                answer = problem.solve()  # 求解逐线路 MILP 或 MISOCP。
            compact_cost.append(np.inf if answer is None else answer['objective'])  # 无解记为无限费用，否则记录已核对的实际投资。
            choices.append(None if answer is None else problem.equations.choice(answer['x']).tolist())  # 把 one-hot 解转为逐线路型号编号，便于人工审阅。
        data['compact_cost'],data['compact_choices'] = compact_cost,choices  # 每种方法保存自己的求解输出，但不复制查询点。
        data['seconds']['compact_point_seconds'] = perf_counter()-start  # 记录紧凑直接模型的投资查询总时间。
        start = perf_counter()  # 开始计量预算内的方向边界查询。
        for budget in budgets:  # 每档预算执行同一组方向查询。
            values = []  # 收集当前预算的所有方向半径。
            for direction in directions:  # 背景负荷固定，仅改变三个独立负荷的比例。
                problem = PlanningModel(equations,budget=budget,direction=direction)  # 直接求解预算和方向约束下的紧凑模型。
                with problem.model:  # 单次边界模型用完即释放。
                    answer = problem.solve()  # 求解最大总负荷及其建设方案。
                values.append(answer['objective'])  # 返回的目标值已经恢复为 kW。
            radii.append(values)  # 按预算顺序保存方向半径。
        data['compact_radii'] = radii  # 保存紧凑模型边界供参考核对。
        data['seconds']['compact_boundary_seconds'] = perf_counter()-start  # 记录与参考口径相同的边界查询时间。
        print(f"{method}: {len(points)} 个投资查询、{len(budgets)*len(directions)} 个边界查询完成",flush=True)  # 输出已经完成的实际查询数量。
        start = perf_counter()  # 联合割循环另行计时，不与直接模型查询混算。
        cuts, queries = [], []  # 历史割跨点和预算复用，查询记录仅引用点或方向索引。
        for i in np.linspace(0,len(points)-1,20,dtype=int):  # 从共享点集中均匀选择 20 个投资查询。
            answer,new,calls = joint_benders(equations,power=points[i],cuts=cuts)  # 仅含 x、p 的 MP 与连续 SP 交替求解。
            cuts.extend(new)  # 全局有效割跨负荷点和预算复用，不保存完整方案表。
            queries.append(dict(kind='point',index=int(i),objective=np.inf if answer is None else answer['objective'],calls=calls))  # 记录投资结论和实际连续 SP 调用次数。
        for j,budget in enumerate(budgets):  # 每档预算另选代表方向验证联合割边界查询。
            for i in (0,1,2,4):  # 三条坐标轴和一条偏斜方向共四条。
                answer,new,calls = joint_benders(equations,budget=budget,direction=directions[i],cuts=cuts,radial_gap_kw=radial_gap_kw)  # 复用历史联合割，按规定 kW 间隙停止。
                cuts.extend(new)  # 只追加本次生成的新割，已有割不重复保存。
                queries.append(dict(kind='ray',budget=j,index=i,objective=answer['objective'],bound=answer['bound'],calls=calls))  # 同时保存可行下界和主问题全局上界。
        data['cuts'],data['joint_queries'] = cuts,queries  # 联合割列表和查询列表分别保存一份。
        data['seconds']['joint_query_seconds'] = perf_counter()-start  # 计时包含本轮 MP、SP 和割管理，不含事后割审核。
        # 在全部 16 个连续方案域上优化检查每一条割，不只看其是否排除了生成点。
        data['cut_minimum'] = verify_joint_cuts(network,method,cuts)  # 每条割均在 16 个独立连续方案域上核对最小余量。
        print(f"{method}: 联合割查询 {len(queries)} 次，全部 {len(cuts)} 条割完成 16 方案连续域核验",flush=True)  # 输出割审核覆盖范围，不能用采样通过替代全域检查。
    record['hashes'] = {p:sha256(Path(p).read_bytes()).hexdigest()  # 记录本次实际参与模型及数据计算的源码指纹。
                        for p in ('model.py','vertify.py','tests/reference.py',*network.sources)}  # 独立消元参考也纳入指纹，便于复现实验。
    record['hashes']['main.ipynb'] = sha256('\n'.join(c.source for c in nbformat.read('main.ipynb',4).cells).encode()).hexdigest()  # Notebook 仅对源单元取指纹，排除运行输出造成的自引用。
    return record  # 返回唯一原始审核记录，汇总表由它现场生成。

**小规模区域基准。** 以下函数只在 `COMPACT_REVIEW=False` 时执行。固定方案与逐线路规划共用 `PlanningEquations` 和 `PlanningSP`，不再保留旧 MP1/MP2 与旧 LP/SOCP 子问题类。

线性基准通过残块搜索和同方案证书构造区域并集。这里的完整方案集合只用于小算例核对；正式非枚举查询由上面的 `joint_benders` 完成。

In [6]:
def linear_region(plans):  # 仅供小规模并集基准；非枚举规划查询使用 joint_benders。
    oracles = [PlanningSP(PlanningEquations(d,'linear',planning=False)) for d in plans]  # 固定方案也复用正式方程及 SP。
    outer = [simplex(d) for d in plans]  # 各方案独立维护候选域，跨方案最终取并集。
    certified, history = {}, []  # 每个内点证书只归属于产生它的方案。
    search = ResidualSearch(outer)  # 从尚未被认证并集覆盖的残块中选择真正顶点。
    while True:  # 所有残块有证书后才停止。
        candidate = search.next(outer,certified)  # 根据同方案凸性检查整块覆盖。
        if candidate is None:  # 没有未覆盖残块，线性并集认证完成。
            break  # 输出各方案记录及完整查询历史。
        i,power = candidate  # 取残块所属方案和待检查负荷。
        answer = oracles[i].solve(np.empty(0),power)  # 固定方案没有规划未知量，x 为空。
        if not answer['feasible'] and answer['cut'] is None:  # 线性顶点查询必须取得可行证书或严格割。
            raise RuntimeError('Linear vertex query is unresolved')  # 不把数值未确定点加入认证并集。
        row = dict(design=int(i),p=power.copy(),eta=answer['eta'],cut=answer['cut'],feasible=answer['feasible'])  # 一次实际 SP 查询只记一条记录。
        history.append(row)  # 回放和调用次数均来自这份历史。
        if answer['cut'] is not None:  # 固定方案联合割退化成四个负荷割系数。
            cut = answer['cut']  # 割方向统一为 a+bᵀp≥0。
            outer[i] = clip_polytope(outer[i],cut[0],cut[1:])  # 只收缩本方案的外域。
        add_certificate(certified,row)  # 仅将原点和获证查询加入对应方案的内域。
    return [dict(design=i,initial=simplex(d),outer=outer[i],inner=certified.get(i,np.zeros((1,3))),  # 同方案的内外域可独立核验。
                 history=[row for row in history if row['design']==i],calls=oracles[i].calls)  # 不在记录中重复保存建设向量及费用。
            for i,d in enumerate(plans)]  # 方案只通过索引引用 Network 的数据。

**统一顶点切割。** 固定网架使用 `planning=False`，规划变量为空。相同 SP 同时服务 LP、SOCP 与混合方法。先认证固定背景下的参数原点；失败查询的状态只用于同方案凸插值，并逐项检查原始约束。

停止条件为 $(1-\tau)U\subseteq I\subseteq D_M\subseteq U$。三个 SP 模式共享这一几何判据；LP 使用 $\tau=0$，SOCP 使用给定径向容差。

In [7]:
def fixed_query(oracle, power, origin):  # 为固定方案构域取得负荷割和同方案可行内点。
    e = oracle.equations  # 固定方案没有建设变量，所有运行系数已确定。
    x = np.empty(0)  # 空选型向量使统一 SP 退化为固定网架查询。
    answer = oracle.solve(x,power)  # 先认证原查询，或取得严格有效的分离割。
    witness, state = np.asarray(power),answer['state']  # 可行时直接采用原查询及其状态。
    alpha = 1.  # 原始约束有严格证书时无需二分。
    if not answer['feasible'] or (e.method=='socp' and e.margin(x,witness,state)<-1e-13):  # SOCP 内域另留锥边界数值余量。
        low,high = 0.,1.  # alpha=0 对应同一固定背景下的可行原点。
        for _ in range(40):  # 二分只用于生成内点，构域停止仍由完整几何包含判据决定。
            alpha = (low+high)/2  # 同时插值负荷和全部运行变量。
            mixed = origin+alpha*(state-origin)  # 固定背景保留在原点状态中，不随负荷缩放。
            if e.margin(x,alpha*witness,mixed)>=-1e-13:  # 所有原始线性约束及二阶锥都满足才移动内侧界。
                low = alpha  # 保存可行的一侧。
            else:  # 违反任一原始约束时只能更新外侧界。
                high = alpha  # 不把二分失败或 eta 当作认证。
        alpha = low  # 只取二分已核验的可行侧。
    if e.method=='socp':  # SOCP 边界点略微内缩，给浮点锥边界留下数值余量。
        alpha *= 1.-1e-8  # 线性精确构域不施加径向近似。
    witness,state = alpha*witness,origin+alpha*(state-origin)  # 返回同方案的可行点及运行证书。
    return dict(eta=answer['eta'],cut=answer['cut'],witness=witness,state=state)  # 内点与查询点分别保存，不能混为一谈。


def cut_region(oracle, initial, tolerance):  # 线性及 SOCP 固定方案共用一个顶点构域循环。
    equations = halfspaces(initial)  # 将候选外域转为 aᵀp+b≤0 的半空间。
    origin = oracle.solve(np.empty(0),np.zeros(3))  # 固定背景下的参数原点必须先获得运行证书。
    if not origin['feasible']:  # 当前内点插值和径向停止判据需要可行原点。
        raise RuntimeError('The fixed parameter origin has no feasibility certificate')  # 不在缺少原点证书时继续构域。
    inner,history = np.zeros((1,3)),[]  # 各方案独立保存已认证内点和实际查询。

    def check(point):  # 一次 SP 查询可以同时更新外域和内域。
        nonlocal inner,equations  # 修改当前固定方案的几何状态。
        answer = fixed_query(oracle,point,origin['state'])  # 所有内点均用原始约束核验。
        inner = np.vstack([inner,answer['witness']])  # 同一方案内点的凸包仍然可行。
        history.append(dict(p=point.copy(),eta=answer['eta'],cut=answer['cut'],witness=answer['witness'].copy()))  # 历史不重复保存运行矩阵。
        if answer['cut'] is not None:  # 仅接受真实分离了当前查询的有效割。
            cut = answer['cut']  # 固定方案的割没有 x 系数。
            equations = np.vstack([equations,-cut[[1,2,3,0]]/np.linalg.norm(cut[1:])])  # 转换为单位法向量的半空间。
        return answer['cut']  # 调用方只在外域变化后重新计算顶点。

    lengths = [np.min(-equations[equations[:,i]>1e-8,3]/equations[equations[:,i]>1e-8,i])  # 初始外域的各轴截距。
               for i in range(3)]  # 三个坐标分别对应三个实际负荷节点。
    for point in .01*np.diag(lengths):  # 先认证三个轴向内点，建立三维内域。
        check(point)  # 查询次数只由正式 SP 统计。
    interior = inner.mean(axis=0)  # 可行轴向内点的平均值用于半空间求交。
    outer = HalfspaceIntersection(equations,interior).intersections  # 只取候选域的几何顶点。
    while True:  # 完整径向包含判据成立后结束，不依赖任意轮数上限。
        inside = halfspaces(inner)  # I 为本方案已经认证的内点凸包。
        gap = np.max((1-tolerance)*outer@inside[:,:3].T+inside[:,3],axis=1)  # 检查 (1-tau)U 的每个顶点是否属于 I。
        if gap.max()<=1e-8:  # 凸性将顶点包含提升为整个固定方案外域的包含。
            return dict(initial=initial,inner=polytope_vertices(inner),outer=outer,history=history,calls=oracle.calls)  # 结果只保存真实凸包顶点和查询记录。
        point = np.maximum(outer[int(np.argmax(gap))],0.)  # 优先检查尚未认证程度最大的外顶点。
        point[point<1e-9] = 0.  # 清除半空间求交产生的极小负轴向舍入残差。
        cut = check(point)  # 取得原查询证书或有效割，并扩充可行内域。
        if cut is None and np.any(inner[-1]<(1-tolerance)*point-1e-8):  # 无可靠割且内点不足时，在目标径向误差内再查询。
            cut = check((1-.5*tolerance)*point)  # 新查询仍使用原始约束完成认证。
        if cut is not None:  # 新增割后才需要更新外域顶点。
            outer = HalfspaceIntersection(equations,interior).intersections  # 求交过程不额外调用潮流模型。

**4．三种构域方法。** 每档预算分别重新求解。混合方法包含自己的线性阶段，计时不借用纯线性结果。所有方案上的外域最后取并集。

In [8]:
def compute_regions(plans, method, tolerance):  # 小规模逐方案基准；所有运行检查均调用统一 PlanningSP。
    start = perf_counter()  # 计时包含固定方程、求解器和几何操作。
    lp_seconds,lp_calls,lp_cuts = 0.,0,0  # 纯 SOCP 没有线性预切阶段。
    if method in ('linear','hybrid'):  # 每种方法独立执行自己的预切，不借用其他方法的计时结果。
        records = linear_region(plans)  # 多方案先用认证并集跳过已经覆盖的残块。
        lp_seconds = perf_counter()-start  # 混合方法总时间已经包含这段耗时。
        lp_calls = sum(r['calls'] for r in records)  # 实际线性 SP 求解次数。
        lp_cuts = sum(h['cut'] is not None for r in records for h in r['history'])  # 只统计实际采用的割。
    if method in ('socp','hybrid'):  # 两种 SOCP 方法使用相同的连续模型和径向误差标准。
        seeds = [r['outer'] for r in records] if method=='hybrid' else [simplex(d) for d in plans]  # 混合方法从每个方案自己的线性外域开始。
        records = [dict(design=i,**cut_region(PlanningSP(PlanningEquations(d,'socp',planning=False)),initial,tolerance))  # 固定选型复用统一的模型方程。
                   for i,(d,initial) in enumerate(zip(plans,seeds))]  # 不将不同方案的内点混成一个凸域。
    timing = dict(region_seconds=perf_counter()-start,linear_stage_seconds=lp_seconds,lp_calls=lp_calls,lp_cuts=lp_cuts)  # 单次构域记录完整时间。
    return records,timing  # 网格归属判定在外层单独计时。

**5．独立 AC 参考与评价。** 完整 AC 保留 $P^2+Q^2=v\ell$，按费用从低到高检查方案。一个点首次得到可行证书时，其费用就是最小可行投资；较便宜方案存在未确定状态时必须继续核实，不能发布错误的最小费用。

AC 每个点只保存一个最小投资，多档预算直接查询，不重复扫描。AC 各预算耗时为本次顺序扫描处理完该预算全部方案的累计实际时间。三种构域方法的总时间包含建模、求解、几何处理和网格判定；公共网格准备、许可证启动及绘图不计。

AC 可行性由独立的等式模型确认；未确定点只有在找到同价或更低价的可行方案时，才不影响该点的最小可行投资。未解决的更便宜方案会阻止输出，避免将求解失败写成不可行标签。

In [9]:
def ac_reference(plans, points, budgets):  # plans 必须按费用升序排列，Network 已保证这一顺序。
    minimum = np.full(len(points), np.inf)  # 尚未找到 AC 可行方案的点暂记为无限费用。
    unknown = np.full(len(points), np.inf)  # 记录未确定方案的最低费用，防止误报最小可行投资。
    times, checks = [], 0  # 保存逐方案累计时间及实际执行的点—方案检查次数。
    environment = gp.Env(empty=True)  # 为可能需要的非凸 AC 核验建立静默求解环境。
    environment.setParam('OutputFlag', 0)  # 关闭共享 AC 求解环境的日志。
    environment.start()  # 许可证启动在 AC 扫描计时之前完成。
    start = perf_counter()  # 计时起点；在方法网格判定段中另起计时，用于构成总时间。
    try:  # 无论扫描是否成功都释放共享环境。
        for i,design in enumerate(plans):  # 从最低投资方案开始逐一检查。
            active = np.flatnonzero(np.isinf(minimum))  # 已经得到更便宜可行方案的点无需再检查高价方案。
            oracle = ACPowerFlow(design)  # 仅读取该方案网架，独立建立完整 AC 递推。
            try:  # 每个方案的非凸 AC 模型拥有独立资源生命周期。
                status = oracle.scan(points[active])  # 批量 AC 三态判定：1 可行、-1 已证不可行、0 未确定。
                checks += len(active)  # 记录实际扫描工作量，不乘以预算数重复统计。
                for j in np.flatnonzero(status==0):  # 仅不动点尚未判定的点需要显式非凸 AC 求解。
                    status[j] = oracle.global_status(points[active[j]], environment)  # 超时或无证书仍是 0，不能当作不可行。
                minimum[active[status==1]] = design.cost  # 费用递增扫描下第一次可行的费用，是候选最小投资。
                unknown[active[status==0]] = np.minimum(unknown[active[status==0]], design.cost)  # 保留未判定的较低费用，最后检查它是否影响最小投资结论。
            finally:  # 完成或中断该方案扫描后都执行释放。
                oracle.close()  # 结束该方案时释放可能创建的非凸模型。
            times.append(perf_counter()-start)  # 包含此前全部方案的累计扫描时间。
            print(f"AC 方案 {i+1}/{len(plans)}：{times[-1]:.1f} s", flush=True)  # 显示真实累计时间，避免大网格运行无反馈。
        if np.any(unknown < minimum):  # 若仍有更便宜的未确定方案，最小投资或不可行性都未被证实。
            raise RuntimeError("Unresolved AC feasibility at a cheaper design")  # 未确定的较低费用方案会阻止报告确定的最小投资。
    finally:  # 退出 AC 扫描时统一释放环境。
        environment.dispose()  # 全部方案处理完后释放共享求解环境。
    costs = np.array([d.cost for d in plans])  # 用方案费用确定每档预算应计到哪一个扫描时刻。
    timings = [dict(region_seconds=times[np.flatnonzero(costs<=b)[-1]], evaluation_seconds=0.)  # AC 各预算耗时是同一次共享扫描的累计值，不是多次独立扫描。
               for b in budgets]  # 各预算按其允许方案的最后一个累计时刻计时。
    return minimum, timings, checks  # 返回每点最小投资、预算计时及真实扫描工作量。


def evaluate(network, designs, budgets, divisions, tolerance):  # 统一评价箱和等体积网格，保证不同方法使用相同体积口径。
    # 评价箱覆盖所有允许方案的线性域，扩容后的区域不会被旧坐标范围截断。
    intercepts = []  # 收集所有允许方案的三个 LP 轴截距，用来包住扩容后的区域。
    for design in designs:  # 为每个允许方案计算评价箱轴向界。
        equations = PlanningEquations(design,'linear',planning=False)  # 同一固定方案的三个轴查询共用方程。
        values = []  # 收集该方案的三条轴向最大负荷。
        for direction in np.eye(3):  # 固定背景不随轴向负荷缩放。
            problem = PlanningModel(equations,direction=direction)  # 直接求连续线性边界作为评价箱依据。
            with problem.model:  # 每条方向查询结束后释放优化模型。
                values.append(problem.solve()['objective'])  # 记录已满足全部线性运行约束的轴截距。
        intercepts.append(values)  # 评价箱取各允许方案轴截距的最大值。
    bounds = np.ceil(np.max(intercepts,axis=0)/10)*10  # 取所有方案截距最大值，并向上整到 10 kW，避免截断规划域。
    grid = np.indices((divisions,)*3, dtype=np.int32).reshape(3,-1).T  # 生成三维单元编号，每行对应一个体素。
    points = (grid+.5)*(bounds/divisions)  # 使用单元中心代表体素；各体素体积相同。
    selected = np.flatnonzero(points.sum(axis=1)<=max(d.power_limit for d in designs))  # 总负荷超出共同有效上界的点直接在所有区域之外。
    points = points[selected]  # 仅对评价箱中尚可能可行的点进行实际判定。
    masks = np.zeros((4,len(budgets),divisions**3), dtype=bool)  # 方法、预算、扁平网格三轴；跳过的点保持域外。
    regions = {m:[] for m in METHODS if m!='ac'}  # 三种切割方法保存几何，AC 用独立网格参考而不伪造凸包。
    timings = {m:[] for m in METHODS}  # 每个方法、每个预算对应一条实测计时记录。
    startup = gp.Model()  # 预先完成公共许可证启动，不把启动延迟只计给第一个方法。
    startup.dispose()  # 预先完成公共许可证启动，不把启动延迟只计给第一个方法。
    with threadpool_limits(limits=1):  # 矩阵运算统一单线程，减少方法比较时线程配置的差异。
        for j,budget in enumerate(budgets):  # 同一预算的三种切割方法使用完全相同的允许方案集合。
            plans = [d for d in designs if d.cost<=budget]  # 实现规划预算约束；各方案单独认证后取并集。
            for method in ('linear','socp','hybrid'):  # 每种方法重新建模并求解，混合方法包含自己的线性阶段。
                records, timing = compute_regions(plans,method,tolerance)  # 计算该预算下的逐方案切割域。
                start = perf_counter()  # 计时起点；在方法网格判定段中另起计时，用于构成总时间。
                masks[METHODS.index(method),j,selected] = region_membership(  # 计算外域并集对同一网格的成员标签。
                    points,[np.asarray(r['outer']) for r in records if len(r['outer'])])  # 空方案域不贡献任何可行点；不能将不同方案合成一个凸包。
                timing['evaluation_seconds'] = perf_counter()-start  # 单独记录网格归属判定时间，再与构域时间相加。
                regions[method].append(records)  # 保存当前方法、当前预算的几何与 SP 记录。
                timings[method].append(timing)  # 时间只保存在元数据中，不在几何记录中复制。
                print(f"预算 {budget:g}, {method}: {timing['region_seconds']+timing['evaluation_seconds']:.3f} s",flush=True)  # 报告包含构域和网格判定的本次耗时。
        minimum, timings['ac'], checks = ac_reference(designs,points,budgets)  # 全部预算共享一遍按费用排序的独立 AC 扫描。
    ac_cost = np.full(divisions**3,np.inf)  # 恢复完整评价箱，之前排除的点保持不可行。
    ac_cost[selected] = minimum  # 每个网格点只保存一个最小可行投资。
    for j,budget in enumerate(budgets):  # 同一预算的三种切割方法使用完全相同的允许方案集合。
        masks[2,j] = np.isfinite(ac_cost)&(ac_cost<=budget)  # 存在预算内可行方案才属于 AC 规划域；inf≤inf 不能误判可行。
    metadata = dict(network=network.name, planning=PLANNING, cost_unit=network.cost_unit,  # 保存 Notebook 的运行设置和 Network 提供的物理语义。
        load_nodes=list(designs[0].load_nodes),bounds=bounds.tolist(),divisions=divisions,  # 保存实际节点、评价箱与网格分辨率。
        budgets=[None if np.isinf(b) else b for b in budgets],radial_tolerance=tolerance,  # 无限预算在 JSON 中用 null 表示。
        sample_count=len(points),timings=timings,point_design_checks=checks,  # 记录评价点数、分方法计时及独立 AC 工作量。
        designs=[dict(x=np.asarray(d.x).tolist(),cost=d.cost) for d in designs],  # 方案向量和费用只保存一张表，几何记录只引用编号。
        hashes={p:sha256(Path(p).read_bytes()).hexdigest() for p in  # 记录本次计算实际使用的模型和网架源码指纹。
                ('model.py','vertify.py','region.py',*network.sources)})  # 对模型、验证、几何及网架来源文件统一取指纹。
    metadata['hashes']['main.ipynb'] = sha256('\n'.join(c.source for c in nbformat.read('main.ipynb',as_version=4).cells).encode()).hexdigest()  # Notebook 指纹只含源单元，排除执行后变化的输出。
    result = BenchmarkResult(masks.reshape((4,len(budgets))+(divisions,)*3),metadata,regions,  # 还原三个空间轴，以同一数据驱动指标、存盘及绘图。
                             ac_cost.reshape((divisions,)*3))  # AC 最小投资恢复成三维数组，供预算域按需生成。
    return result  # 返回一次完整实验的唯一结果容器。

**6．执行实验。** 上面的函数定义就是本实验使用的全部流程。重新运行此单元从头计算，不暗中续用另一次实验的求解结果。

In [10]:
if COMPACT_REVIEW:  # 紧凑模型审核与完整小算例构域使用同一个入口。
    output = Path('results')/CASE/'compact_review'  # 模型等价性审核使用独立输出目录。
    output.mkdir(parents=True,exist_ok=True)  # 创建该次审核的结果目录。
    if RECOMPUTE:  # 显式选择重新计算时才运行全部优化查询。
        with threadpool_limits(limits=1):  # 所有方法采用相同的矩阵运算线程数。
            review = review_compact(network,budgets)  # 完成投资、边界和联合割连续域核验。
        np.savez_compressed(output/'review.npz',record=json.dumps(review,default=lambda x:x.tolist(),ensure_ascii=False))  # 整次审核仅保存一个原始记录文件，不重复落盘汇总表。
    else:  # 读取模式直接加载现有审核记录。
        with np.load(output/'review.npz',allow_pickle=False) as data:  # 禁止对象反序列化，仅恢复数组中的 JSON 文本。
            review = json.loads(str(data['record']))  # 汇总和显示均使用加载的同一份原始记录。
else:  # 另一实验模式完成区域构造与独立 AC 对比。
    if RECOMPUTE:  # 显式重新计算时，从当前源码和网架开始完整实验。
        validation = validate_power_flow(designs[0]) if CASE=='case33bw' else None  # 33 节点先做独立节点导纳矩阵潮流交叉核验。
        result = evaluate(network,designs,budgets,DIVISIONS,RADIAL_TOLERANCE)  # 执行切割、独立 AC、网格成员判定及计时。
        result.metadata['validation'] = validation  # 将交叉核验依据并入本次唯一结果记录。
        result.save(OUTPUT)  # 整次实验只在此保存一次原始结果。
    else:  # 未要求重算时读取已有区域实验结果。
        result = BenchmarkResult.load(OUTPUT)  # 明确读取此前结果，不把旧时间标成当前重新求解时间。

Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2685996


Academic license 2685996 - for non-commercial use only - registered to 20___@mail.scut.edu.cn


linear: 124 个投资查询、48 个边界查询完成


linear: 联合割查询 36 次，全部 17 条割完成 16 方案连续域核验


socp: 124 个投资查询、48 个边界查询完成


socp: 联合割查询 36 次，全部 51 条割完成 16 方案连续域核验


**7．FR、MR 与求解时间。** $FR=\mathrm{Vol}(\widehat D\setminus D_{AC})/\mathrm{Vol}(\widehat D)$，$MR=\mathrm{Vol}(D_{AC}\setminus\widehat D)/\mathrm{Vol}(D_{AC})$。均在同一网格上计算，网格零误差不代表连续误差严格为零。

In [11]:
if COMPACT_REVIEW:  # 紧凑审核模式展示相同查询上的等价性与边界误差。
    compact_summary = pd.DataFrame(compact_review_summary(review))  # 从唯一原始记录计算汇总表。
    display(compact_summary.T)  # 转置后逐项对比线性与 SOCP 两列。
    # 下列断言属于本次模型等价性实验，不进入通用求解器。
    assert (compact_summary.minimum_cost_mismatches==0).all()  # 所有审核点的最低投资应与枚举参考相同。
    assert (compact_summary.budget_membership_mismatches==0).all()  # 四档预算下的区域归属都应一致。
    assert (compact_summary.joint_cost_mismatches==0).all()  # 联合割查询应得到相同的最小投资。
    assert (compact_summary.max_radius_difference_kw<.002).all()  # 直接模型与参考边界的最大差必须小于 0.002 kW。
    assert (compact_summary.joint_max_radius_difference_kw<.002).all()  # 联合割边界采用相同的绝对误差标准。
    assert (compact_summary.joint_max_gap_kw<=review['radial_gap_kw']+1e-5).all()  # 可行下界与全局上界的间隙不得超过设定精度及数值余量。
    assert (compact_summary.minimum_cut_margin>=-1e-7).all()  # 全部联合割的连续域最小余量须通过数值核验。
else:  # 区域实验模式展示 FR/MR 和完整计算时间。
    summary = pd.DataFrame(result.summary)  # FR/MR 从保存的基础标签现场推导，不维护第二份指标文件。
    summary['method'] = summary['method'].map(dict(zip(METHODS,METHOD_NAMES)))  # 仅替换显示名称，不修改方法轴和原始记录。
    display(summary[['budget','method','fr_percent','mr_percent','total_seconds']].rename(columns={  # 只展示用户关心的预算、方法、FR/MR 和时间。
        'budget':f'预算 ({network.cost_unit})','method':'方法','fr_percent':'FR (%)',  # 按网架投资单位给表头标注物理含义。
        'mr_percent':'MR (%)','total_seconds':'总计算时间 (s)'}).style.format(precision=5))  # 百分比与秒统一保留五位小数用于比较。

,0,1
method,linear,socp
point_count,124,124
minimum_cost_mismatches,0,0
budget_membership_mismatches,0,0
radial_queries,48,48
max_radius_difference_kw,0.000001,0.000077
joint_queries,36,36
joint_cost_mismatches,0,0
joint_max_radius_difference_kw,0.001,0.001
joint_max_gap_kw,0.001,0.001001


**8．交互图与切割回放。** 蓝色为与 AC 重合，红色遗漏，黄色多余；同预算四幅图同步旋转。回放从已保存的 SP 查询记录重建固定方案的内外域，不调用求解器。

In [12]:
if not COMPACT_REVIEW:  # 完整三维对比图仅从已完成的区域实验生成。
    page = save_method_comparison(result,OUTPUT)  # 图形与 FR/MR 消费同一网格标签，红色遗漏、黄色多余。
    replay = save_replay(result,OUTPUT)  # 从真实 SP 历史重建切割过程，不额外求解表面网格点。
    display(IFrame(src=page.as_posix(),width='100%',height=1220))  # 嵌入可旋转的四方法对比页面。
    display(IFrame(src=replay.as_posix(),width='100%',height=740))  # 嵌入基于真实 SP 历史的切割回放页面。